Tarea 1

instalar:
pip install opencv-python mediapipe numpy

In [20]:

import cv2
import mediapipe as mp
import numpy as np
import random

# Filtro de gafas de sol. Si se hace una o con la boca sale humo.

# ==========================================
# 1. CONFIGURACIÓN DEL DETECTOR FACIAL
# ==========================================

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

# Cargar gafas
gafas = cv2.imread("gafas.webp", cv2.IMREAD_UNCHANGED)

if gafas is None:
    raise FileNotFoundError(
        "Comprueba que gafas.webp está en la carpeta del notebook"
    )
    

# ==========================================
# 2. FUNCIÓN PARA COLOCAR LAS GAFAS
# ==========================================


def poner_gafas(frame, imagen, centro, ancho, angulo):

    alto = int(imagen.shape[0] * ancho / imagen.shape[1])

    overlay = cv2.resize(imagen, (ancho, alto))

    # Asegurar que la imagen tiene canales de color
    if overlay.ndim == 2:
        overlay = cv2.cvtColor(
            overlay, cv2.COLOR_GRAY2BGR
        )

    # Si no tiene transparencia, añadir un canal alfa
    if overlay.shape[2] == 3:
        overlay = cv2.cvtColor(
            overlay, cv2.COLOR_BGR2BGRA
        )

    # Crear un lienzo ampliado para evitar recortes al girar
    diagonal = int(np.sqrt(ancho**2 + alto**2)) + 10

    lienzo = np.zeros(
        (diagonal, diagonal, 4), dtype=np.uint8
    )

    # Centrar las gafas en el lienzo
    x_inicio = (diagonal - ancho) // 2
    y_inicio = (diagonal - alto) // 2

    lienzo[
        y_inicio:y_inicio + alto,
        x_inicio:x_inicio + ancho
    ] = overlay

    # Girar las gafas sin recortar sus extremos
    matriz = cv2.getRotationMatrix2D(
        (diagonal // 2, diagonal // 2),
        angulo,
        1
    )

    overlay = cv2.warpAffine(
        lienzo,
        matriz,
        (diagonal, diagonal),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    # Posición del accesorio
    x = int(centro[0] - diagonal / 2)
    y = int(centro[1] - diagonal / 2)

    h, w = frame.shape[:2]

    # Recortar únicamente los bordes de la pantalla
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(w, x + diagonal)
    y2 = min(h, y + diagonal)

    if x1 >= x2 or y1 >= y2:
        return frame

    accesorio = overlay[
        y1-y:y2-y,
        x1-x:x2-x
    ]

    region = frame[y1:y2, x1:x2]

    # Mezcla mediante transparencia
    alpha = accesorio[:, :, 3:4] / 255.0

    frame[y1:y2, x1:x2] = (
        accesorio[:, :, :3] * alpha +
        region * (1 - alpha)
    ).astype(np.uint8)

    return frame



def generar_humo(frame, particulas, centro_boca, activar):

    # Crear nuevas partículas cuando la boca tenga forma de O
    if activar:
        for _ in range(3):
            particulas.append({
                "x": centro_boca[0] + random.randint(-5, 5),
                "y": centro_boca[1],
                "radio": random.randint(5, 9),
                "vida": random.randint(25, 40),
                "desplazamiento": random.uniform(-1.5, 1.5)
            })

    overlay = np.zeros_like(frame)

    # Actualizar las partículas existentes
    for p in particulas[:]:

        p["y"] -= 2
        p["x"] += p["desplazamiento"]
        p["radio"] += 0.15
        p["vida"] -= 1

        if p["vida"] <= 0:
            particulas.remove(p)
            continue

        # Dibujar cada partícula
        cv2.circle(
            overlay,
            (int(p["x"]), int(p["y"])),
            int(p["radio"]),
            (190, 190, 190),
            -1
        )

    # Difuminar para conseguir aspecto de humo
    overlay = cv2.GaussianBlur(overlay, (31, 31), 0)

    # Mezclar el humo con la imagen original
    frame = cv2.addWeighted(
        frame, 1.0,
        overlay, 0.35,
        0
    )

    return frame



# ==========================================
# 3. INICIAR WEBCAM
# ==========================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():

    print("No se ha podido acceder a la cámara")

else:

    with FaceLandmarker.create_from_options(options) as landmarker:

        frame_index = 0
        particulas_humo = []
        
        while True:

            ret, frame = cap.read()

            if not ret:
                print("No se ha podido leer la imagen")
                break

            # Espejo
            frame = cv2.flip(frame, 1)

            height, width, _ = frame.shape

            # Preparar imagen para MediaPipe
            rgb = cv2.cvtColor(
                frame, cv2.COLOR_BGR2RGB
            )

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            timestamp_ms = frame_index * 33

            results = landmarker.detect_for_video(
                mp_image, timestamp_ms
            )

            # ==========================================
            # 4. DETECCIÓN Y COLOCACIÓN DE LAS GAFAS
            # ==========================================

            if results.face_landmarks:

                for face in results.face_landmarks:
                    
                    # Extremos de los ojos
                    ojo_izq = (
                        int(face[33].x * width),
                        int(face[33].y * height)
                    )

                    ojo_der = (
                        int(face[263].x * width),
                        int(face[263].y * height)
                    )

                    # Laterales de la cara (sienes)
                    sien_izq = (
                        int(face[127].x * width),
                        int(face[127].y * height)
                    )

                    sien_der = (
                        int(face[356].x * width),
                        int(face[356].y * height)
                    )

                    # Centro de las gafas
                    centro = (
                        (ojo_izq[0] + ojo_der[0]) // 2,
                        (ojo_izq[1] + ojo_der[1]) // 2
                    )

                    # Ancho según la distancia entre las sienes
                    distancia_sienes = np.linalg.norm(
                        np.array(sien_der) -
                        np.array(sien_izq)
                    )

                    ancho_gafas = int(
                        distancia_sienes * 1.3
                    )

                    # Inclinación de la cabeza
                    angulo = -np.degrees(np.arctan2(
                        ojo_der[1] - ojo_izq[1],
                        ojo_der[0] - ojo_izq[0]
                    ))

                    # Colocar las gafas una sola vez
                    frame = poner_gafas(
                        frame,
                        gafas,
                        centro,
                        ancho_gafas,
                        angulo
                    )
                    
                    #######################################
                    
                    
                    # ==========================================
                    # DETECCIÓN DE LA BOCA EN FORMA DE O
                    # ==========================================

                    # Extremos de la boca
                    boca_izq = np.array([
                        face[61].x * width,
                        face[61].y * height
                    ])

                    boca_der = np.array([
                        face[291].x * width,
                        face[291].y * height
                    ])

                    # Labio superior e inferior
                    labio_sup = np.array([
                        face[13].x * width,
                        face[13].y * height
                    ])

                    labio_inf = np.array([
                        face[14].x * width,
                        face[14].y * height
                    ])

                    # Calcular anchura y apertura de la boca
                    ancho_boca = np.linalg.norm(boca_der - boca_izq)

                    alto_boca = np.linalg.norm(labio_inf - labio_sup)

                    # Relación entre apertura y anchura
                    ratio_boca = alto_boca / (ancho_boca + 1e-6)

                    # Centro de la boca
                    centro_boca = (
                        int((boca_izq[0] + boca_der[0]) / 2),
                        int((boca_izq[1] + boca_der[1]) / 2)
                    )

                    # Detectar boca abierta y redondeada
                    boca_en_o = ratio_boca > 0.15

                    # Generar humo
                    frame = generar_humo(
                        frame,
                        particulas_humo,
                        centro_boca,
                        boca_en_o
                    )


            # Mostrar resultado
            cv2.imshow(
                "FaceFilters - Gafas de sol",
                frame
            )

            frame_index += 1

            # Pulsar Q para salir
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()
